# Actual KV-Cache Compression Benchmark Notebook (FullKV / MiniCache / CommonKV / ThinkKV / PaLu)

This notebook now applies compression on **actual KV cache tensors** (`past_key_values`) during autoregressive decoding.

Implemented strategies in one framework:
- `fullkv` (uncompressed baseline)
- `minicache`
- `commonkv`
- `thinkkv`
- `palu`

And evaluates them in one run on LongBench + Ruler style data.

In [ ]:
# !pip install -U torch transformers datasets accelerate sentencepiece pandas numpy tqdm matplotlib

import json
import math
import time
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

## 1) Load model/tokenizer (Transformers; real KV cache path)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"   # change as needed
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading model:", MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
    trust_remote_code=True,
)
if DEVICE == "cpu":
    model = model.to(DEVICE)
model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

## 2) KV cache compression policies (tensor-level, not prompt-level)

In [ ]:
class KVCompressor:
    name = "base"

    def reset(self):
        pass

    def step(self, past_key_values, layer_attn_last_token=None):
        """Compress past_key_values and return new past_key_values.

        past_key_values: tuple(num_layers) of (K, V)
          K/V shape typically [B, H, T, D]
        layer_attn_last_token: list/tuple with per-layer attention weights for newest token
          each tensor shape [B, H, T]
        """
        return past_key_values


class FullKV(KVCompressor):
    name = "fullkv"


class MiniCache(KVCompressor):
    """Paper-inspired: sink + local window + uniform anchors over older context."""
    name = "minicache"

    def __init__(self, sink_tokens=32, local_tokens=512, anchor_tokens=128):
        self.sink_tokens = sink_tokens
        self.local_tokens = local_tokens
        self.anchor_tokens = anchor_tokens

    def _indices(self, T, device):
        if T <= self.sink_tokens + self.local_tokens + self.anchor_tokens:
            return torch.arange(T, device=device)
        sink = torch.arange(0, self.sink_tokens, device=device)
        local_start = max(self.sink_tokens, T - self.local_tokens)
        local = torch.arange(local_start, T, device=device)

        old_start = self.sink_tokens
        old_end = local_start
        if old_end <= old_start:
            anchors = torch.empty(0, dtype=torch.long, device=device)
        else:
            anchors = torch.linspace(old_start, old_end - 1, steps=self.anchor_tokens, device=device).long().unique()

        idx = torch.cat([sink, anchors, local]).unique(sorted=True)
        return idx

    def step(self, past_key_values, layer_attn_last_token=None):
        new_past = []
        for (k, v) in past_key_values:
            # [B,H,T,D]
            T = k.shape[2]
            idx = self._indices(T, k.device)
            new_past.append((k.index_select(2, idx), v.index_select(2, idx)))
        return tuple(new_past)


class CommonKV(KVCompressor):
    """Paper-inspired dedup: remove redundant old tokens by K-similarity within each layer/head."""
    name = "commonkv"

    def __init__(self, sink_tokens=32, local_tokens=512, sim_threshold=0.98, max_old_tokens=512):
        self.sink_tokens = sink_tokens
        self.local_tokens = local_tokens
        self.sim_threshold = sim_threshold
        self.max_old_tokens = max_old_tokens

    def _compress_head(self, k, v):
        # k,v: [T,D] for one head
        T = k.shape[0]
        if T <= self.sink_tokens + self.local_tokens:
            return k, v

        sink = torch.arange(0, self.sink_tokens, device=k.device)
        local_start = max(self.sink_tokens, T - self.local_tokens)
        local = torch.arange(local_start, T, device=k.device)
        old = torch.arange(self.sink_tokens, local_start, device=k.device)

        if old.numel() == 0:
            idx = torch.cat([sink, local])
            return k.index_select(0, idx), v.index_select(0, idx)

        old_k = k.index_select(0, old)
        old_v = v.index_select(0, old)

        # if too many old tokens, subsample first for speed
        if old_k.shape[0] > self.max_old_tokens:
            sample_idx = torch.linspace(0, old_k.shape[0] - 1, self.max_old_tokens, device=k.device).long()
            old_k = old_k.index_select(0, sample_idx)
            old_v = old_v.index_select(0, sample_idx)

        norm = torch.nn.functional.normalize(old_k, dim=-1)
        sim = norm @ norm.transpose(0, 1)
        keep = torch.ones(sim.shape[0], dtype=torch.bool, device=k.device)
        for i in range(sim.shape[0]):
            if not keep[i]:
                continue
            dup = (sim[i] > self.sim_threshold)
            dup[: i + 1] = False
            keep[dup] = False

        old_keep = torch.where(keep)[0]
        old_k2 = old_k.index_select(0, old_keep)
        old_v2 = old_v.index_select(0, old_keep)

        k_new = torch.cat([k.index_select(0, sink), old_k2, k.index_select(0, local)], dim=0)
        v_new = torch.cat([v.index_select(0, sink), old_v2, v.index_select(0, local)], dim=0)
        return k_new, v_new

    def step(self, past_key_values, layer_attn_last_token=None):
        out = []
        for (k, v) in past_key_values:
            B, H, T, D = k.shape
            k_heads, v_heads = [], []
            for h in range(H):
                kh, vh = self._compress_head(k[0, h], v[0, h])
                k_heads.append(kh.unsqueeze(0))
                v_heads.append(vh.unsqueeze(0))
            # align lengths by minimum across heads
            min_t = min(x.shape[1] for x in k_heads)
            k_cat = torch.stack([x[:, :min_t, :] for x in k_heads], dim=1)  # [1,H,T,D]
            v_cat = torch.stack([x[:, :min_t, :] for x in v_heads], dim=1)
            out.append((k_cat, v_cat))
        return tuple(out)


class ThinkKV(KVCompressor):
    """Attention-aware retention: keep sink+local and top-k by accumulated attention importance."""
    name = "thinkkv"

    def __init__(self, sink_tokens=32, local_tokens=512, global_budget=1024):
        self.sink_tokens = sink_tokens
        self.local_tokens = local_tokens
        self.global_budget = global_budget
        self.importance = None  # list per layer: [B,H,T]

    def reset(self):
        self.importance = None

    def step(self, past_key_values, layer_attn_last_token=None):
        # layer_attn_last_token should match current cache length before compression
        if layer_attn_last_token is not None:
            if self.importance is None:
                self.importance = [a.detach().clone() for a in layer_attn_last_token]
            else:
                new_imp = []
                for imp, a in zip(self.importance, layer_attn_last_token):
                    T_old = imp.shape[-1]
                    T_new = a.shape[-1]
                    if T_new > T_old:
                        pad = torch.zeros(*imp.shape[:-1], T_new - T_old, device=imp.device, dtype=imp.dtype)
                        imp = torch.cat([imp, pad], dim=-1)
                    imp = imp[..., :T_new] + a
                    new_imp.append(imp)
                self.importance = new_imp

        out = []
        new_importance = []
        for li, (k, v) in enumerate(past_key_values):
            B, H, T, D = k.shape
            if T <= self.global_budget:
                out.append((k, v))
                if self.importance is not None:
                    new_importance.append(self.importance[li][..., :T])
                continue

            sink = torch.arange(0, min(self.sink_tokens, T), device=k.device)
            local_start = max(self.sink_tokens, T - self.local_tokens)
            local = torch.arange(local_start, T, device=k.device)

            base = torch.cat([sink, local]).unique(sorted=True)
            remaining = max(0, self.global_budget - base.numel())
            middle_start = sink.numel()
            middle_end = local_start

            if remaining > 0 and middle_end > middle_start and self.importance is not None:
                imp = self.importance[li].mean(dim=(0, 1))  # [T]
                cand = torch.arange(middle_start, middle_end, device=k.device)
                vals = imp.index_select(0, cand)
                topk = min(remaining, cand.numel())
                pick_rel = torch.topk(vals, k=topk).indices
                picked = cand.index_select(0, pick_rel)
            else:
                picked = torch.empty(0, dtype=torch.long, device=k.device)

            idx = torch.cat([base, picked]).unique(sorted=True)
            out.append((k.index_select(2, idx), v.index_select(2, idx)))

            if self.importance is not None:
                new_importance.append(self.importance[li].index_select(-1, idx))

        if self.importance is not None:
            self.importance = new_importance
        return tuple(out)


class PaLu(KVCompressor):
    """Low-rank projection over old KV blocks + exact sink/local (paper-inspired tensor compression)."""
    name = "palu"

    def __init__(self, sink_tokens=32, local_tokens=512, rank=64, block=256):
        self.sink_tokens = sink_tokens
        self.local_tokens = local_tokens
        self.rank = rank
        self.block = block

    def _compress_matrix(self, x):
        # x: [T,D], project to rank-r and reconstruct compressed representatives by block
        T, D = x.shape
        if T <= self.block or self.rank >= min(T, D):
            return x
        # centered low-rank SVD
        m = x.mean(dim=0, keepdim=True)
        xc = x - m
        try:
            U, S, Vh = torch.linalg.svd(xc, full_matrices=False)
            r = min(self.rank, U.shape[1])
            xr = (U[:, :r] * S[:r]) @ Vh[:r, :] + m
        except Exception:
            xr = x
        # block pooling to reduce token count while preserving projected info
        chunks = []
        for i in range(0, xr.shape[0], self.block):
            chunks.append(xr[i:i+self.block].mean(dim=0, keepdim=True))
        return torch.cat(chunks, dim=0)

    def step(self, past_key_values, layer_attn_last_token=None):
        out = []
        for (k, v) in past_key_values:
            B, H, T, D = k.shape
            if T <= self.sink_tokens + self.local_tokens:
                out.append((k, v))
                continue

            sink = torch.arange(0, self.sink_tokens, device=k.device)
            local_start = max(self.sink_tokens, T - self.local_tokens)
            local = torch.arange(local_start, T, device=k.device)
            old = torch.arange(self.sink_tokens, local_start, device=k.device)

            k_heads, v_heads = [], []
            for h in range(H):
                k_sink = k[0, h].index_select(0, sink)
                v_sink = v[0, h].index_select(0, sink)
                k_old = k[0, h].index_select(0, old)
                v_old = v[0, h].index_select(0, old)
                k_loc = k[0, h].index_select(0, local)
                v_loc = v[0, h].index_select(0, local)

                k_old_c = self._compress_matrix(k_old)
                v_old_c = self._compress_matrix(v_old)

                k_new = torch.cat([k_sink, k_old_c, k_loc], dim=0)
                v_new = torch.cat([v_sink, v_old_c, v_loc], dim=0)
                k_heads.append(k_new.unsqueeze(0))
                v_heads.append(v_new.unsqueeze(0))

            min_t = min(x.shape[1] for x in k_heads)
            k_cat = torch.stack([x[:, :min_t, :] for x in k_heads], dim=1)
            v_cat = torch.stack([x[:, :min_t, :] for x in v_heads], dim=1)
            out.append((k_cat, v_cat))
        return tuple(out)


COMPRESSORS = {
    "fullkv": FullKV(),
    "minicache": MiniCache(),
    "commonkv": CommonKV(),
    "thinkkv": ThinkKV(),
    "palu": PaLu(),
}

print("Compressors:", list(COMPRESSORS.keys()))

## 3) Generation with real KV-cache compression

In [ ]:
@torch.no_grad()
def generate_with_kv_compression(
    model,
    tokenizer,
    prompt: str,
    compressor: KVCompressor,
    max_new_tokens: int = 128,
    temperature: float = 0.0,
    top_p: float = 1.0,
):
    compressor.reset()

    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    out = model(**inputs, use_cache=True, output_attentions=True)
    past = out.past_key_values

    # init attention importance with prefill last-token attention
    layer_attn = [a[:, :, -1, :].detach() for a in out.attentions] if out.attentions is not None else None
    past = compressor.step(past, layer_attn_last_token=layer_attn)

    generated = []
    next_logits = out.logits[:, -1, :]

    for _ in range(max_new_tokens):
        if temperature > 0:
            probs = torch.softmax(next_logits / temperature, dim=-1)
            if top_p < 1.0:
                sorted_probs, sorted_idx = torch.sort(probs, descending=True)
                csum = torch.cumsum(sorted_probs, dim=-1)
                mask = csum > top_p
                mask[..., 1:] = mask[..., :-1].clone()
                mask[..., 0] = False
                sorted_probs[mask] = 0
                sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
                idx = torch.multinomial(sorted_probs, num_samples=1)
                next_token = sorted_idx.gather(-1, idx)
            else:
                next_token = torch.multinomial(probs, num_samples=1)
        else:
            next_token = torch.argmax(next_logits, dim=-1, keepdim=True)

        token_id = next_token.item()
        if token_id == tokenizer.eos_token_id:
            break
        generated.append(token_id)

        out = model(
            input_ids=next_token,
            past_key_values=past,
            use_cache=True,
            output_attentions=True,
        )
        past = out.past_key_values
        layer_attn = [a[:, :, -1, :].detach() for a in out.attentions] if out.attentions is not None else None
        past = compressor.step(past, layer_attn_last_token=layer_attn)

        next_logits = out.logits[:, -1, :]

    text = tokenizer.decode(generated, skip_special_tokens=True)
    return text

## 4) Metrics + benchmark loaders (LongBench / Ruler)

In [ ]:
TOKEN_RE = re.compile(r"\w+", re.UNICODE)

def tok_words(s: str):
    return TOKEN_RE.findall((s or "").lower())


def exact_match(pred, refs):
    p = (pred or "").strip().lower()
    return float(any(p == (r or "").strip().lower() for r in refs))


def f1_score(pred, refs):
    p = tok_words(pred)
    if not p:
        return 0.0
    best = 0.0
    for r in refs:
        t = tok_words(r)
        if not t:
            continue
        cnt = {}
        for x in t:
            cnt[x] = cnt.get(x, 0) + 1
        c = 0
        for x in p:
            if cnt.get(x, 0) > 0:
                c += 1
                cnt[x] -= 1
        if c == 0:
            continue
        pr = c / len(p)
        rc = c / len(t)
        best = max(best, 2 * pr * rc / (pr + rc))
    return best


def rouge_l(pred, refs):
    # lightweight approximation via F1 overlap fallback
    return f1_score(pred, refs)


def score(pred, refs, metric):
    m = metric.lower()
    if m in {"em", "exact_match"}: return exact_match(pred, refs)
    if m in {"f1", "qa_f1"}: return f1_score(pred, refs)
    if m in {"rouge", "rouge_l"}: return rouge_l(pred, refs)
    return f1_score(pred, refs)


@dataclass
class Example:
    eid: str
    benchmark: str
    group: str
    task: str
    prompt: str
    refs: List[str]
    metric: str


def load_longbench(task_specs, split="test", limit_per_task=50):
    from datasets import load_dataset
    out = []
    for spec in task_specs:
        task = spec["task"]
        ds = load_dataset("THUDM/LongBench", task, split=split)
        if limit_per_task:
            ds = ds.select(range(min(limit_per_task, len(ds))))
        for i, ex in enumerate(ds):
            prompt = ex.get("prompt") or ex.get("input") or ""
            if not prompt and "context" in ex and "question" in ex:
                prompt = f"Context:
{ex['context']}

Question: {ex['question']}
Answer:"
            refs = ex.get("answers") or ex.get("answer") or ex.get("output") or [""]
            if not isinstance(refs, list):
                refs = [str(refs)]
            out.append(Example(
                eid=f"longbench::{task}::{i}",
                benchmark="LongBench",
                group=spec["group"],
                task=task,
                prompt=str(prompt),
                refs=[str(x) for x in refs],
                metric=spec.get("metric", "f1"),
            ))
    return out


def load_ruler_jsonl(root_dir, task_specs, limit_per_task=50):
    out = []
    root = Path(root_dir)
    for spec in task_specs:
        task = spec["task"]
        fp = root / f"{task}.jsonl"
        if not fp.exists():
            print(f"[WARN] Missing: {fp}")
            continue
        with fp.open("r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if limit_per_task and i >= limit_per_task:
                    break
                ex = json.loads(line)
                prompt = ex.get("prompt") or ex.get("input") or ""
                refs = ex.get("answers") or ex.get("answer") or ex.get("output") or [""]
                if not isinstance(refs, list):
                    refs = [str(refs)]
                out.append(Example(
                    eid=f"ruler::{task}::{i}",
                    benchmark="Ruler",
                    group=spec["group"],
                    task=task,
                    prompt=str(prompt),
                    refs=[str(x) for x in refs],
                    metric=spec.get("metric", "em"),
                ))
    return out

## 5) Unified eval runner (all methods, all tasks, one table)

In [ ]:
def run_eval(examples: List[Example], methods: List[str], max_new_tokens=128, temperature=0.0):
    rows = []
    total = len(examples) * len(methods)

    for m in methods:
        COMPRESSORS[m].reset()

    with tqdm(total=total, desc="Eval") as pbar:
        for ex in examples:
            for m in methods:
                comp = COMPRESSORS[m]
                t0 = time.perf_counter()
                err = ""
                try:
                    pred = generate_with_kv_compression(
                        model=model,
                        tokenizer=tokenizer,
                        prompt=ex.prompt,
                        compressor=comp,
                        max_new_tokens=max_new_tokens,
                        temperature=temperature,
                    )
                except Exception as e:
                    pred = ""
                    err = str(e)
                lat = time.perf_counter() - t0
                sc = 0.0 if err else score(pred, ex.refs, ex.metric)

                rows.append({
                    "eid": ex.eid,
                    "benchmark": ex.benchmark,
                    "group": ex.group,
                    "task": ex.task,
                    "method": m,
                    "metric": ex.metric,
                    "score": sc,
                    "latency_s": lat,
                    "error": err,
                })
                pbar.update(1)

    return pd.DataFrame(rows)


def build_commonkv_style_table(df: pd.DataFrame):
    piv = df.pivot_table(index="method", columns=["benchmark", "group"], values="score", aggfunc="mean")
    piv.columns = [f"{b}:{g}" for b, g in piv.columns]
    avg = df.groupby("method")["score"].mean().rename("Avg.")
    lat = df.groupby("method")["latency_s"].mean().rename("Latency(s)")
    out = piv.join(avg).join(lat)
    return out.sort_values("Avg.", ascending=False)

## 6) Task groups (matching your requested CommonKV-style columns)

In [ ]:
LONGBENCH_TASK_SPECS = [
    {"task": "narrativeqa", "group": "QA", "metric": "f1"},
    {"task": "qasper", "group": "QA", "metric": "f1"},
    {"task": "gov_report", "group": "Sum.", "metric": "rouge_l"},
    {"task": "multi_news", "group": "Sum.", "metric": "rouge_l"},
    {"task": "trec", "group": "FShot", "metric": "em"},
    {"task": "hotpotqa", "group": "Synth.", "metric": "f1"},
    {"task": "lcc", "group": "Code", "metric": "em"},
]

RULER_TASK_SPECS = [
    {"task": "ns", "group": "NS", "metric": "em"},
    {"task": "nmk", "group": "NMK", "metric": "em"},
    {"task": "nmq", "group": "NMQ", "metric": "em"},
    {"task": "nmv", "group": "NMV", "metric": "em"},
    {"task": "qa", "group": "QA", "metric": "f1"},
    {"task": "vt", "group": "VT", "metric": "em"},
    {"task": "fwe", "group": "FWE", "metric": "em"},
]

## 7) Load data and run all methods

In [ ]:
LONG_LIMIT = 20     # set None for full run
RULER_LIMIT = 20    # set None for full run
RULER_ROOT = "./ruler_data"

examples = []
examples += load_longbench(LONGBENCH_TASK_SPECS, split="test", limit_per_task=LONG_LIMIT)
examples += load_ruler_jsonl(RULER_ROOT, RULER_TASK_SPECS, limit_per_task=RULER_LIMIT)

print("Total examples:", len(examples))

METHODS = ["fullkv", "minicache", "commonkv", "thinkkv", "palu"]
results_df = run_eval(examples, METHODS, max_new_tokens=128, temperature=0.0)
summary_table = build_commonkv_style_table(results_df)
summary_table

## 8) Save outputs

In [ ]:
out = Path("artifacts")
out.mkdir(exist_ok=True)
results_df.to_csv(out / "kvcache_eval_raw.csv", index=False)
summary_table.to_csv(out / "kvcache_eval_summary.csv")
print("Saved artifacts/kvcache_eval_raw.csv and artifacts/kvcache_eval_summary.csv")